# Task 44: Manual Ensemble Voting

Calculate hard and soft votes directly from scratch classifier probabilities. Derive normalized model weights from validation precision, recall, or F1, then compare ensemble and base performance on a held-out test split.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

candidates=[Path('heart.csv'),Path('Task_44_Manual_Ensemble_Voting/heart.csv'),Path('Heart_Disease_Tasks/Task_44_Manual_Ensemble_Voting/heart.csv')]
data=pd.read_csv(next(p for p in candidates if p.is_file())).drop_duplicates().reset_index(drop=True)
cols=[c for c in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'] if c in data]
X,y=data[cols].to_numpy(float),data.target.to_numpy(int)
rng=np.random.default_rng(44)
def split(labels,fraction,seed):
    generator=np.random.default_rng(seed);a=[];b=[]
    for label in np.unique(labels):
        ids=np.flatnonzero(labels==label);generator.shuffle(ids);cut=round(fraction*len(ids));a.extend(ids[:cut]);b.extend(ids[cut:])
    generator.shuffle(a);generator.shuffle(b);return np.array(a),np.array(b)
trainval,test=split(y,.8,44);train_local,val_local=split(y[trainval],.75,45);train,val=trainval[train_local],trainval[val_local]
Xtr,Xv,Xte=X[train],X[val],X[test];ytr,yv,yte=y[train],y[val],y[test]
mean,std=Xtr.mean(0),Xtr.std(0);std[std==0]=1
Xtr,Xv,Xte=(Xtr-mean)/std,(Xv-mean)/std,(Xte-mean)/std


def sigmoid(z):return 1/(1+np.exp(-np.clip(z,-35,35)))
class Logistic:
    def fit(self,x,y):
        self.w=np.zeros(x.shape[1]);self.b=0.
        for _ in range(1100):
            p=sigmoid(x@self.w+self.b);e=p-y;self.w-=.06*(x.T@e/len(y)+.02*self.w);self.b-=.06*e.mean()
        return self
    def proba(self,x):return sigmoid(x@self.w+self.b)
class Tree:
    def __init__(self,depth=4):self.depth=depth
    def fit(self,x,y):self.root=self.grow(x,y,0);return self
    def grow(self,x,y,d):
        counts=np.bincount(y,minlength=2);node={'prob':float(y.mean())}
        if d>=self.depth or len(y)<8 or len(np.unique(y))<2:return node
        parent=1-np.sum((counts/len(y))**2);best=None
        for j in range(x.shape[1]):
            unique=np.unique(x[:,j])
            for t in (unique[:-1]+unique[1:])/2:
                m=x[:,j]<=t
                if not m.any() or m.all():continue
                lc=np.bincount(y[m],minlength=2)/m.sum();rc=np.bincount(y[~m],minlength=2)/(~m).sum()
                gain=parent-m.mean()*(1-np.sum(lc**2))-(~m).mean()*(1-np.sum(rc**2))
                if best is None or gain>best[0]:best=(gain,j,t,m)
        if best is None or best[0]<=0:return node
        _,j,t,m=best;node.update(j=j,t=t,left=self.grow(x[m],y[m],d+1),right=self.grow(x[~m],y[~m],d+1));return node
    def proba(self,x):
        out=[]
        for row in x:
            node=self.root
            while 'j' in node:node=node['left'] if row[node['j']]<=node['t'] else node['right']
            out.append(node['prob'])
        return np.array(out)
class Forest:
    def fit(self,x,y,count=7):
        rng=np.random.default_rng(47);self.trees=[]
        for _ in range(count):
            ids=rng.integers(0,len(y),len(y));self.trees.append(Tree(4).fit(x[ids],y[ids]))
        return self
    def proba(self,x):return np.mean([tree.proba(x) for tree in self.trees],axis=0)
class Nearest:
    def __init__(self,k=7):self.k=k
    def fit(self,x,y):self.x,self.y=x,y;return self
    def proba(self,x):
        result=[]
        for row in x:
            distances=np.sqrt(np.sum((self.x-row)**2,axis=1));ids=np.argsort(distances,kind='stable')[:self.k]
            result.append(float(self.y[ids].mean()))
        return np.array(result)

def scores(actual,pred):
    tp=np.sum((actual==1)&(pred==1));fp=np.sum((actual==0)&(pred==1));fn=np.sum((actual==1)&(pred==0));tn=np.sum((actual==0)&(pred==0))
    precision=tp/(tp+fp) if tp+fp else 0.;recall=tp/(tp+fn) if tp+fn else 0.
    return {'accuracy':(tp+tn)/len(actual),'precision':precision,'recall':recall,'f1':2*precision*recall/(precision+recall) if precision+recall else 0.}

models={'Logistic':Logistic().fit(Xtr,ytr),'Decision tree':Tree(4).fit(Xtr,ytr),'Random forest':Forest().fit(Xtr,ytr),'k-NN':Nearest(7).fit(Xtr,ytr)}
validation={name:model.proba(Xv) for name,model in models.items()}
validation_f1={name:scores(yv,(p>=.5).astype(int))['f1'] for name,p in validation.items()}
weights=np.array([validation_f1[name] for name in models]);weights=weights/weights.sum()
probability_matrix=np.vstack([model.proba(Xte) for model in models.values()])

def vote(probabilities,weights,mode='soft'):
    if mode=='soft':combined=weights@probabilities
    else:
        hard=(probabilities>=.5).astype(float);combined=weights@hard
        ties=np.isclose(combined,.5);combined[ties]=(weights@probabilities[:,ties])
    return (combined>=.5).astype(int),combined

results=[]
for name,model in models.items():results.append({'Method':name,**scores(yte,(model.proba(Xte)>=.5).astype(int))})
for mode in ['hard','soft']:
    prediction,_=vote(probability_matrix,weights,mode)
    results.append({'Method':f'{mode.title()} vote weighted by validation F1',**scores(yte,prediction)})
display(pd.DataFrame(results).set_index('Method').round(3))
print('Validation F1-derived weights:',dict(zip(models.keys(),weights.round(3))))
print('Hard-vote ties use weighted probability as a deterministic tie-break. Validation sets weights; test data is used only for final metrics.')

,accuracy,precision,recall,f1
Method,,,,
Logistic,0.852,0.816,0.939,0.873
Decision tree,0.803,0.818,0.818,0.818
Random forest,0.787,0.794,0.818,0.806
k-NN,0.803,0.784,0.879,0.829
Hard vote weighted by validation F1,0.852,0.833,0.909,0.870
Soft vote weighted by validation F1,0.852,0.816,0.939,0.873


Validation F1-derived weights: {'Logistic': np.float64(0.246), 'Decision tree': np.float64(0.256), 'Random forest': np.float64(0.242), 'k-NN': np.float64(0.256)}
Hard-vote ties use weighted probability as a deterministic tie-break. Validation sets weights; test data is used only for final metrics.
